# Baseline: TF-IDF + linear models

Word and character n-gram TF-IDF with LinearSVC / LogisticRegression. Fixed validation split from the EDA (`splits/train_folds.csv`, fold 0 = validation). Vectorizers are fitted on the training folds only. Metric: macro-F1. Seed fixed at 42.

In [22]:
import time
from functools import lru_cache
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import ParameterGrid
from sklearn.svm import LinearSVC
from tqdm.auto import tqdm

SEED = 42
np.random.seed(SEED)

LABELS = ["human", "ai", "fusion"]
VAL_FOLD = 0
SPLITS = ("train", "val")

pd.set_option("display.max_colwidth", 80)
sns.set_theme(style="whitegrid", rc={"axes.spines.top": False, "axes.spines.right": False})

In [23]:
DATA_DIR = Path(kagglehub.competition_download("ai-generated-text-detection-hw3-ucu-ai-2026"))

train = pd.read_csv(DATA_DIR / "train.csv").merge(pd.read_csv("../splits/train_folds.csv"), on="id", validate="1:1")
test = pd.read_csv(DATA_DIR / "test.csv")

train_fold = train[train["fold"] != VAL_FOLD].reset_index(drop=True)
val_fold = train[train["fold"] == VAL_FOLD].reset_index(drop=True)
y_val = val_fold["label"].to_numpy()

print(f"train folds: {len(train_fold):,} | val (fold {VAL_FOLD}): {len(val_fold):,} | test: {len(test):,}")
pd.DataFrame({"train %": train_fold["label"].value_counts(normalize=True),
              "val %": val_fold["label"].value_counts(normalize=True)}).loc[LABELS].mul(100).round(1).T

train folds: 245,116 | val (fold 0): 27,236 | test: 25,000


label,human,ai,fusion
train %,24.5,44.8,30.7
val %,24.5,44.8,30.7


## 1. Fixed search sample
The parameter search runs on a fixed 50k subsample of the training folds (stratified by label × language, seed 42), always scored on the full validation fold. The best configuration is refitted on as many training-fold texts as memory allows in section 4.

In [13]:
N_SEARCH = 50_000


def stratified_sample(df, n, seed=SEED):
    """Fixed random sample of n rows with the label × language shares of df (all rows if n >= len(df))."""
    if n >= len(df):
        return df.reset_index(drop=True)
    strata = df["label"] + "|" + df["lang"]
    return (df.groupby(strata, group_keys=False)
              .sample(frac=n / len(df), random_state=seed)
              .reset_index(drop=True))


search = stratified_sample(train_fold, N_SEARCH)
y_search = search["label"].to_numpy()

print(f"search sample: {len(search):,} texts")
(search["label"] + " | " + search["lang"]).value_counts(normalize=True).mul(100).round(1).rename("search %").to_frame().T

search sample: 50,000 texts


,ai | en-EN,fusion | en-EN,human | en-EN,ai | de-DE,fusion | de-DE,human | de-DE
search %,37.3,24.1,19.2,7.5,6.6,5.3


## 2. Vectorizers
Two TF-IDF views, both fitted on the search sample only and then applied to the validation fold:
- **word** 1–2-grams, lowercased;
- **char** 3–5-grams inside word boundaries (`char_wb`), case kept: typos, lowercase "i" and source artefacts such as `(CNN)` are not cleaned away (EDA, section 3).

`sublinear_tf` dampens repeated terms in long texts; `min_df=3` and `max_features` keep the vocabulary manageable.

In [14]:
TFIDF_PARAMS = {
    "word": dict(analyzer="word", ngram_range=(1, 2), lowercase=True, min_df=3, max_features=300_000,
                 sublinear_tf=True, dtype=np.float32),
    "char": dict(analyzer="char_wb", ngram_range=(3, 5), lowercase=False, min_df=3, max_features=300_000,
                 sublinear_tf=True, dtype=np.float32),
}


def build_tfidf(view, train_text):
    """Fit the TF-IDF vectorizer of one view on train-fold texts only; return it with the train matrix."""
    vec = TfidfVectorizer(**TFIDF_PARAMS[view])
    X_train = vec.fit_transform(tqdm(train_text, desc=f"{view}: fit", unit="text"))
    return vec, X_train


vectorizers, features, vec_stats = {}, {}, []
for view in TFIDF_PARAMS:
    t0 = time.time()
    vectorizers[view], X_train = build_tfidf(view, search["text"])
    X_val = vectorizers[view].transform(tqdm(val_fold["text"], desc=f"{view}: val", unit="text"))
    features[view] = {"train": X_train, "val": X_val}
    vec_stats.append({"view": view, "features": X_train.shape[1], "nnz / text": X_train.nnz // X_train.shape[0],
                      "seconds": round(time.time() - t0)})
vec_stats = pd.DataFrame(vec_stats).set_index("view")

word: fit:   0%|          | 0/50000 [00:00<?, ?text/s]

word: val:   0%|          | 0/27236 [00:00<?, ?text/s]

char: fit:   0%|          | 0/50000 [00:00<?, ?text/s]

char: val:   0%|          | 0/27236 [00:00<?, ?text/s]

In [15]:
vec_stats

,features,nnz / text,seconds
view,,,
word,300000,530,38
char,300000,3015,116


## 3. Parameter search
Grid on the search sample, every configuration scored on the full validation fold:
- features: word, char, word + char (stacked side by side);
- LinearSVC with C ∈ {0.1, 0.3, 1}; LogisticRegression with C ∈ {3, 10, 30} (`RUN_MODELS` selects which ones run);
- class weights: none vs `balanced` (macro-F1 weighs the 25% human class as much as the 45% ai class).

In [ ]:
FEATURE_SETS = ["word", "char", "word+char"]
MODELS = {"LinearSVC": LinearSVC, "LogReg": LogisticRegression}
GRID = {
    "LinearSVC": {"C": [0.1, 0.3, 1], "class_weight": [None, "balanced"]},
    "LogReg": {"C": [3, 10, 30], "class_weight": [None, "balanced"]},
}
FIXED_PARAMS = {"LinearSVC": {}, "LogReg": {"max_iter": 1000}}
RUN_MODELS = ["LogReg"]  # which models the grid runs; LinearSVC results: commit 31e9846


def make_model(model_name, params):
    """Unfitted model with its grid params plus fixed params and the seed."""
    return MODELS[model_name](**params, **FIXED_PARAMS[model_name], random_state=SEED)


@lru_cache(maxsize=1)  # configs are ordered by feature set: only the current stacked matrices stay in memory
def feature_set(name):
    """Train/val matrices of a feature set; "word+char" stacks the two views side by side."""
    views = name.split("+")
    if len(views) == 1:
        return features[views[0]]
    return {split: sparse.hstack([features[v][split] for v in views]).tocsr() for split in SPLITS}


def run_one(feat_name, model_name, params):
    """Fit one config on the search sample; return its validation scores and the fitted model."""
    X = feature_set(feat_name)
    t0 = time.time()
    clf = make_model(model_name, params).fit(X["train"], y_search)
    pred = clf.predict(X["val"])
    per_class = f1_score(y_val, pred, labels=LABELS, average=None)
    row = {"features": feat_name, "model": model_name, "C": params["C"],
           "class_weight": params["class_weight"] or "none",
           "macro_f1": f1_score(y_val, pred, average="macro"),
           **{f"f1_{label}": f1 for label, f1 in zip(LABELS, per_class)},
           "seconds": round(time.time() - t0)}
    return row, clf


configs = [(f, m, p) for f in FEATURE_SETS for m in RUN_MODELS for p in ParameterGrid(GRID[m])]
rows, fitted = [], []
bar = tqdm(configs, desc="grid", unit="fit")
for feat_name, model_name, params in bar:
    bar.set_postfix_str(f"{feat_name} | {model_name} C={params['C']} cw={params['class_weight'] or 'none'}")
    row, clf = run_one(feat_name, model_name, params)
    rows.append(row)
    fitted.append(clf)
    bar.set_description(f"grid (best macro-F1 {max(r['macro_f1'] for r in rows):.4f})")
feature_set.cache_clear()  # free the stacked word+char matrices

results = pd.DataFrame(rows)  # results row i <-> fitted[i]

grid:   0%|          | 0/18 [00:00<?, ?fit/s]

In [ ]:
results.sort_values("macro_f1", ascending=False).round(4)

,features,model,C,class_weight,macro_f1,f1_human,f1_ai,f1_fusion,seconds
17,word+char,LinearSVC,1.0,balanced,0.8599,0.7940,0.9831,0.8026,31
16,word+char,LinearSVC,1.0,none,0.8595,0.7931,0.9826,0.8027,30
15,word+char,LinearSVC,0.3,balanced,0.8592,0.7974,0.9830,0.7973,20
14,word+char,LinearSVC,0.3,none,0.8584,0.7948,0.9821,0.7982,20
13,word+char,LinearSVC,0.1,balanced,0.8545,0.7977,0.9810,0.7846,17
12,word+char,LinearSVC,0.1,none,0.8528,0.7934,0.9789,0.7862,18
9,char,LinearSVC,0.3,balanced,0.8521,0.7895,0.9796,0.7874,17
11,char,LinearSVC,1.0,balanced,0.8507,0.7816,0.9798,0.7908,24
8,char,LinearSVC,0.3,none,0.8505,0.7855,0.9783,0.7878,18
10,char,LinearSVC,1.0,none,0.8495,0.7784,0.9794,0.7907,23


## 4. Refit and submission
The best grid configuration is refitted on as many training-fold texts as memory allows (stratified by label × language, seed 42), scored again on the validation fold, and used to predict the test set.
- The validation fold stays out of the refit, so the submitted model has an honest validation score and nothing is fitted on validation rows.
- The section-2 vectorizers are reused unchanged (vocabulary and IDF from the search sample, so training folds only). Only the classifier is refitted.
- `MEM_BUDGET_GB` caps the training size. LinearSVC copies the data to float64 and then to liblinear's own format (~36 bytes per non-zero vs 8 for the float32 matrix). With char n-grams (~3k non-zeros per text) all 245k texts do not fit in 16 GB. With a larger budget (more RAM) the cap reaches all training folds.

In [ ]:
MEM_BUDGET_GB = 5  # rough cap for the training matrix and solver copies; raise it on a machine with more RAM
BYTES_PER_NNZ = {"LinearSVC": 8 + 12 + 16,  # float32 CSR + float64 copy + liblinear feature_node
                 "LogReg": 8 + 8}           # float32 CSR (kept as is by lbfgs), twice while chunks are stacked


def max_train_texts(feat_name, model_name, budget_gb=MEM_BUDGET_GB):
    """Number of training texts whose matrix and solver copies fit in budget_gb (estimated on the search sample)."""
    nnz_per_text = sum(features[v]["train"].nnz for v in feat_name.split("+")) / len(search)
    return int(budget_gb * 1e9 / (nnz_per_text * BYTES_PER_NNZ[model_name]))


def featurize(feat_name, texts, desc, chunk=10_000):
    """Transform texts with the section-2 vectorizers, chunk by chunk to limit peak memory."""
    texts = list(texts)
    views = feat_name.split("+")
    parts = [sparse.hstack([vectorizers[v].transform(texts[i:i + chunk]) for v in views]).tocsr()
             for i in tqdm(range(0, len(texts), chunk), desc=desc, unit="chunk")]
    return sparse.vstack(parts).tocsr()


def refit(config, n_texts):
    """Fit a grid config on a stratified sample of n_texts training-fold texts."""
    final_train = stratified_sample(train_fold, n_texts)
    X = featurize(config["features"], final_train["text"], "refit: train")
    params = {"C": config["C"], "class_weight": None if config["class_weight"] == "none" else config["class_weight"]}
    return make_model(config["model"], params).fit(X, final_train["label"].to_numpy())


best = results.loc[results["macro_f1"].idxmax()]
n_final = min(len(train_fold), max_train_texts(best["features"], best["model"]))
if n_final > len(search):
    final_clf = refit(best, n_final)
else:  # the budget allows no more than the search sample: keep the grid model
    final_clf, n_final = fitted[best.name], len(search)

val_pred = final_clf.predict(feature_set(best["features"])["val"])
X_test = featurize(best["features"], test["text"], "test")
feature_set.cache_clear()

final_summary = pd.DataFrame([{
    **best[["features", "model", "C", "class_weight"]].to_dict(),
    "train texts": n_final,
    "macro_f1 (search fit)": best["macro_f1"],
    "macro_f1 (final)": f1_score(y_val, val_pred, average="macro"),
}])

In [19]:
final_summary.round(4)

,features,model,C,class_weight,train texts,macro_f1 (search fit),macro_f1 (final)
0,word+char,LinearSVC,1.0,balanced,50000,0.8599,0.8599


In [20]:
SUB_DIR = Path("../submissions")


def make_submission(ids, labels, sample):
    """Predictions in sample-submission order, checked against the format rules in COMPETITION.md."""
    sub = sample[["id"]].merge(pd.DataFrame({"id": ids, "label": labels}), on="id", how="left", validate="1:1")
    assert len(sub) == 25_000 and sub["id"].equals(sample["id"]), "rows or order differ from sample_submission.csv"
    assert sub["label"].isin(LABELS).all(), "missing or unknown labels"
    return sub


sub = make_submission(test["id"], final_clf.predict(X_test), pd.read_csv(DATA_DIR / "sample_submission.csv"))

SUB_DIR.mkdir(exist_ok=True)
sub_path = SUB_DIR / (f"baseline_{best['features'].replace('+', '_')}_{best['model']}"
                      f"_C{best['C']:g}_cw-{best['class_weight']}.csv")
sub.to_csv(sub_path, index=False)
print(f"saved {sub_path} ({len(sub):,} rows)")

saved ../submissions/baseline_word_char_LinearSVC_C1_cw-balanced.csv (25,000 rows)


In [21]:
# predicted class shares on test should look like the validation ones
pd.DataFrame({"val true %": pd.Series(y_val).value_counts(normalize=True),
              "val pred %": pd.Series(val_pred).value_counts(normalize=True),
              "test pred %": sub["label"].value_counts(normalize=True)}).loc[LABELS].mul(100).round(1).T

,human,ai,fusion
val true %,24.5,44.8,30.7
val pred %,27.2,45.2,27.6
test pred %,26.8,52.0,21.3
